# RNA3D Pilot10 | One-click GPU VRAM profiling — 3 models × 10 sequences

**Purpose.** Re-run **trRosettaRNA2, DRfold2 and Protenix** with the same frozen input conditions as the original exploratory pilot, measure **observed peak GPU memory** for *each* target, validate every prediction, and immediately save its evidence to the existing `MyDrive/RNA3D_Benchmark` project. **No TM-score/RMSD recomputation** occurs in this notebook.

### How to use

1. Open this notebook in **Google Colab**, and select **Runtime → Change runtime type → T4 GPU** (the model originally used for all three pilot runs).
2. Click **Runtime → Run all**. Authorize the Google Drive mount once.
3. The notebook automatically runs **trRosettaRNA2 → DRfold2 → Protenix**, processing all 10 targets for each. **There are no manual batch toggles or review stops.**
4. Progress and results are saved after each target. If Colab disconnects, reopen the same notebook on a T4 and click **Run all**; verified profiling results are skipped and incomplete attempts are retained as evidence rather than overwritten.

**Outputs:** `MyDrive/RNA3D_Benchmark/Evaluations/gpu_memory_pilot10_20261121_v1/peak_gpu_memory_30_targets.csv` plus a readable Markdown report, individual model `.../runs/*_vram_20261121_v1/<target>/complete.json` records and per-attempt `gpu_memory_trace.csv.gz`, `peak_vram.json`, inference logs and selected prediction.

**Measurement:** NVIDIA NVML sampled **total device memory used** during a single inference, every ~200 ms, with idle baseline and a baseline-subtracted field. This is a *sampled observed peak*, not a guaranteed instantaneous hardware maximum and not a process-exclusive CUDA allocator peak. Only use one active inference at a time and leave the GPU otherwise idle. Do not compare raw peaks from different GPU hardware. Profiles are evidence for the pilot; they do **not** certify template scarcity or training-set independence.

**Reproducibility:** original reference runs remain untouched. Frozen upstream commits and input modes are recorded. Failures save diagnostics; an incomplete 30/30 panel is never labeled complete. Large model weights are installed in temporary Colab storage, not written into Drive.


In [ ]:
# Cell 1/6 — automatic Drive mount, fixed experiment manifest and NVML preflight
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, csv, json, time, math, gzip, hashlib, shutil, subprocess, datetime
import threading, traceback, signal, tempfile, urllib.request, tarfile, io, importlib.util, platform

ROOT = Path('/content/drive/MyDrive/RNA3D_Benchmark')
INPUTS = ROOT/'NuFold/runs/pilot10_single_20261018T203832Z/inputs'
EVAL = ROOT/'Evaluations/gpu_memory_pilot10_20261121_v1'
TARGETS = ['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B','9lq7_K',
           '8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
EXPECTED = [36,71,59,51,33,49,41,45,61,66]
SAMPLING_INTERVAL_SECONDS = 0.20
MIN_ACTIVE_GPU_DELTA_MIB = 64.0  # reject CPU fallback or a GPU-empty inference trace
REQUIRE_T4 = True
RUNS = {
    'trRosettaRNA2': {
        'folder': 'trRosettaRNA2', 'run_id': 'trrosettarna2_pilot10_vram_20261121_v1',
        'original_run': 'trrosettarna2_pilot10_single_20261114_v1',
        'pin': 'd40a40e36617cb4004a11b0a7603167415ed9c52',
        'mode': 'single_sequence_a3m_depth_1_internal_ss',
        'prediction_filename': 'model_1_relaxed200.pdb',
    },
    'DRfold2': {
        'folder': 'DRfold2', 'run_id': 'drfold2_pilot10_vram_20261121_v1',
        'original_run': 'drfold2_pilot10_20261110_v1',
        'pin': '3667d6ac44c5b14cff94c36e0371224acb2cdae5',
        'mode': 'upstream_default_single_output_fasta',
        'prediction_filename': 'model_1.pdb',
    },
    'Protenix': {
        'folder': 'Protenix', 'run_id': 'protenix_pilot10_vram_20261121_v1',
        'original_run': 'protenix_pilot10_rna_only_nomsa_notemplate_v2_layernormtorch',
        'pin': '85767b811c40ed46e73a9b39519cf6bfca8701ba',
        'mode': 'rna_only_no_msa_no_templates_fp32_layernormtorch_seed101_cycle10_step200',
        'prediction_filename': 'prediction.cif',
    },
}
MODEL_ORDER = ['trRosettaRNA2','DRfold2','Protenix']
assert ROOT.is_dir() and INPUTS.is_dir(), 'Missing mounted MyDrive/RNA3D_Benchmark frozen inputs.'
assert len(TARGETS)==len(EXPECTED)==len(set(TARGETS))==10

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

def fasta_sequence(path):
    lines=[l.strip() for l in Path(path).read_text().splitlines() if l.strip()]
    assert lines and lines[0].startswith('>') and sum(l.startswith('>') for l in lines)==1, str(path)
    seq=''.join(l for l in lines if not l.startswith('>')).upper().replace('T','U')
    assert seq and set(seq)<=set('ACGU'), str(path)
    return seq

def a3m_first_sequence(path):
    seq=[]; seen=False
    for ln in Path(path).read_text().splitlines():
        ln=ln.strip()
        if ln.startswith('>'):
            if seen: break
            seen=True
        elif seen: seq.append(ln)
    assert seen
    return ''.join(x for x in ''.join(seq) if x not in '.-' and not x.islower()).upper().replace('T','U')

INPUT={}
for target, expected in zip(TARGETS,EXPECTED):
    fasta=INPUTS/target/(target+'.fasta')
    a3m=INPUTS/target/(target+'.a3m')
    assert fasta.is_file() and a3m.is_file(), f'Missing frozen input(s): {target}'
    seq=fasta_sequence(fasta)
    assert len(seq)==expected, f'Frozen length mismatch {target}: {len(seq)} vs {expected}'
    assert a3m_first_sequence(a3m)==seq, f'A3M and FASTA disagree: {target}'
    assert sum(l.startswith('>') for l in a3m.read_text().splitlines())==1, f'A3M MSA depth != 1: {target}'
    INPUT[target]={'length':expected,'seq':seq,'fasta':fasta,'a3m':a3m,
                   'fasta_sha256':sha256(fasta),'a3m_sha256':sha256(a3m)}
print('Frozen input validation: 10/10 targets; A3M depth = 1')

# NVML measures subprocess GPU allocation as well as the notebook process.
if importlib.util.find_spec('pynvml') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','nvidia-ml-py'],check=True)
import pynvml
pynvml.nvmlInit()
assert pynvml.nvmlDeviceGetCount()>=1, 'No NVIDIA GPU available; select a T4 GPU runtime.'
GPU_HANDLE=pynvml.nvmlDeviceGetHandleByIndex(0)
GPU_NAME=pynvml.nvmlDeviceGetName(GPU_HANDLE)
if isinstance(GPU_NAME,bytes): GPU_NAME=GPU_NAME.decode()
GPU_TOTAL_MIB=round(pynvml.nvmlDeviceGetMemoryInfo(GPU_HANDLE).total/(1024**2),2)
if REQUIRE_T4:
    assert 'T4' in GPU_NAME, f'Expected a Tesla T4 like the source pilot, got {GPU_NAME}. Change Colab runtime GPU, then Run all.'
print('GPU:',GPU_NAME,'| total VRAM MiB:',GPU_TOTAL_MIB)
print('Profiler interval:',int(SAMPLING_INTERVAL_SECONDS*1000),'ms')
EVAL.mkdir(parents=True,exist_ok=True)


In [ ]:
# Cell 2/6 — robust GPU sampler, per-target verification and Drive-safe storage

def run_checked(args,cwd=None,log_file=None):
    print('$', ' '.join(str(a) for a in args)[:350],flush=True)
    if log_file is None:
        subprocess.run(list(map(str,args)),cwd=cwd,check=True)
    else:
        Path(log_file).parent.mkdir(parents=True,exist_ok=True)
        with Path(log_file).open('a') as f:
            p=subprocess.run(list(map(str,args)),cwd=cwd,stdout=f,stderr=subprocess.STDOUT)
        if p.returncode:
            raise RuntimeError(f'Setup command failed ({p.returncode}): {args}. See {log_file}.')

def json_load(path):
    return json.loads(Path(path).read_text())

def json_save(path,data,exclusive=False):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('x' if exclusive else 'w') as f:
        json.dump(data,f,indent=2,allow_nan=False)
        f.write('\n')

def run_root(model):
    cfg=RUNS[model]
    return ROOT/cfg['folder']/'runs'/cfg['run_id']

def lock_model_run(model):
    root=run_root(model)
    root.mkdir(parents=True,exist_ok=True)
    lock=root/'profiling_protocol.json'
    config=dict(model=model,pin=RUNS[model]['pin'],mode=RUNS[model]['mode'],
                gpu_name=GPU_NAME,gpu_total_mib=GPU_TOTAL_MIB,
                targets=TARGETS,lengths=EXPECTED,
                fasta_sha256={t:INPUT[t]['fasta_sha256'] for t in TARGETS},
                a3m_sha256={t:INPUT[t]['a3m_sha256'] for t in TARGETS},
                sample_interval_seconds=SAMPLING_INTERVAL_SECONDS,
                measurement='nvml_device_memory_used_sampled')
    if lock.exists():
        old=json_load(lock)
        assert all(old.get(k)==v for k,v in config.items()), (
            f'Profiling run already exists with different input/model/GPU settings: {lock}. '
            'Use the same T4 and frozen inputs or choose a new run ID.')
    else:
        json_save(lock,dict(config,created_utc=utc_now()),exclusive=True)
    return root

def prior_output_hash(model, target):
    cfg=RUNS[model]
    folder=ROOT/cfg['folder']/'runs'/cfg['original_run']/target
    p=folder/('inference.json' if model=='trRosettaRNA2' else 'complete.json')
    if not p.is_file(): return None
    try:
        meta=json_load(p)
        if model=='trRosettaRNA2': return meta.get('output_pdb_sha256')
        if model=='DRfold2': return meta.get('pdb_sha256')
        for a in meta.get('artifacts',[]):
            if a.get('filename')=='prediction.cif': return a.get('sha256')
    except Exception: pass
    return None

class GPUTrace:
    """Device-level NVML sampler, includes any CUDA child processes spawned by CLI."""
    def __init__(self,trace_path,interval=SAMPLING_INTERVAL_SECONDS):
        self.trace_path=Path(trace_path)
        self.interval=interval
        self.stop_event=threading.Event()
        self.errors=[]
        self.n=0
        self.peak=-1
        self.started=time.monotonic()
        self.baseline_samples=[]
        self.thread=None
        self._gzip=None
    def _used_mib(self):
        return round(pynvml.nvmlDeviceGetMemoryInfo(GPU_HANDLE).used/(1024**2),3)
    def start(self):
        self.baseline_samples=[self._used_mib() for _ in range(3)]
        self.baseline_mib=max(self.baseline_samples)
        self.started=time.monotonic()
        self._gzip=gzip.open(self.trace_path,'wt',encoding='utf-8',newline='')
        self.writer=csv.writer(self._gzip)
        self.writer.writerow(['seconds_since_start','gpu_device_memory_used_mib'])
        self.thread=threading.Thread(target=self._loop,daemon=True)
        self.thread.start()
    def _loop(self):
        while not self.stop_event.is_set():
            try:
                v=self._used_mib()
                self.peak=max(self.peak,v)
                self.writer.writerow([round(time.monotonic()-self.started,4),v])
                self.n+=1
            except Exception as ex:
                self.errors.append(repr(ex))
                break
            self.stop_event.wait(self.interval)
    def finish(self):
        self.stop_event.set()
        if self.thread: self.thread.join(timeout=5)
        if self.thread and self.thread.is_alive():
            raise RuntimeError('GPU sampler thread did not terminate cleanly')
        try:
            if not self.errors:
                v=self._used_mib()
                self.peak=max(self.peak,v)
                self.writer.writerow([round(time.monotonic()-self.started,4),v])
                self.n+=1
        finally:
            if self._gzip: self._gzip.close()
        if self.errors: raise RuntimeError('NVML sampler errors: '+str(self.errors[:3]))
        assert self.n>=2 and self.peak>=0, 'Missing GPU telemetry; target cannot be marked PASS.'
        return dict(peak_gpu_memory_mib_observed=self.peak,
                    baseline_gpu_memory_mib=self.baseline_mib,
                    peak_gpu_memory_mib_above_baseline=round(max(0,self.peak-self.baseline_mib),3),
                    sampling_interval_ms=int(self.interval*1000),sample_count=self.n,
                    measurement_method='NVML sampled total device-memory used; includes CUDA child processes',
                    gpu_name=GPU_NAME,gpu_total_memory_mib=GPU_TOTAL_MIB,
                    trace_sha256=sha256(self.trace_path))

def execute_monitored(command,cwd,local_log,local_trace,env=None):
    trace=GPUTrace(local_trace)
    proc=None
    start=time.monotonic()
    with Path(local_log).open('w') as log:
        trace.start()
        try:
            proc=subprocess.Popen(list(map(str,command)),cwd=str(cwd),env=env,
                                  stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
            return_code=proc.wait()
        except BaseException:
            if proc is not None and proc.poll() is None:
                try: os.killpg(proc.pid,signal.SIGTERM)
                except ProcessLookupError: pass
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    try: os.killpg(proc.pid,signal.SIGKILL)
                    except ProcessLookupError: pass
                    proc.wait()
            raise
        finally:
            telemetry=trace.finish()
    return return_code,round(time.monotonic()-start,3),telemetry

def validate_pdb(p,expected_len):
    p=Path(p)
    assert p.is_file() and p.stat().st_size>150, f'Prediction missing: {p}'
    residues=set(); atoms=0
    for line in p.open(errors='replace'):
        if line.startswith(('ATOM  ','HETATM')) and len(line)>=54:
            if line[16] not in ' A1': continue
            atoms+=1
            for a,b in [(30,38),(38,46),(46,54)]:
                assert math.isfinite(float(line[a:b])), 'Non-finite PDB coordinates'
            residues.add((line[21],line[22:27]))
    assert atoms>=20 and len(residues)==expected_len, f'PDB residues {len(residues)} != {expected_len}, atoms {atoms}'
    return {'residues':len(residues),'atoms':atoms}

def validate_cif(p,conf,expected_len):
    import gemmi
    obj=gemmi.read_structure(str(p))
    assert len(obj)>=1
    residues=[r for chain in obj[0] for r in chain if r.entity_type==gemmi.EntityType.Polymer]
    assert len(residues)==expected_len, f'CIF residues {len(residues)} != {expected_len}'
    atoms=[atom for res in residues for atom in res]
    assert atoms and all(all(math.isfinite(v) for v in (a.pos.x,a.pos.y,a.pos.z)) for a in atoms), 'Invalid/NaN CIF atoms'
    confidence=json_load(conf)
    for key in ['plddt','ptm','ranking_score']:
        assert isinstance(confidence.get(key),(int,float)) and math.isfinite(confidence[key]), f'Invalid {key}'
    return {'residues':len(residues),'atoms':len(atoms),
            'confidence':{k:float(confidence[k]) for k in ('plddt','ptm','ranking_score')}}

def verified_complete(model,target):
    folder=run_root(model)/target
    marker=folder/'complete.json'
    if not marker.exists(): return None
    try:
        data=json_load(marker)
        assert data['status']=='PASS' and data['model']==model and data['target_id']==target
        assert data['model_commit']==RUNS[model]['pin']
        assert data['fasta_sha256']==INPUT[target]['fasta_sha256']
        assert data['gpu_name']==GPU_NAME
        assert data['peak_gpu_memory_mib_observed']>0
        assert data['sample_count']>=2
        attempt=folder/data['attempt']
        for artifact in data['artifacts']:
            file=attempt/artifact['filename']
            assert file.is_file() and sha256(file)==artifact['sha256'], f'Corrupt artifact {file}'
        return data
    except (OSError,AssertionError,ValueError,TypeError,KeyError,RuntimeError) as e:
        print('INVALID completion marker (not skipping):',model,target,str(e)[:240])
        return None

def next_attempt(folder):
    folder.mkdir(parents=True,exist_ok=True)
    k=1
    while (folder/f'attempt_{k:03d}').exists():
        k+=1
    sub=folder/f'attempt_{k:03d}'
    sub.mkdir(parents=True,exist_ok=False)
    return sub

def prepare_command(model,target,scratch):
    cfg=RUNS[model]
    src=INPUT[target]
    if model=='trRosettaRNA2':
        envpy=Path('/content/micromamba_root/envs/trRNA2/bin/python')
        args=[str(envpy),'-m','trRNA2.predict','-i',str(src['a3m']),'-o',str(scratch/'prediction')]
        return args,Path('/content/trRosettaRNA2'),os.environ.copy()
    if model=='DRfold2':
        args=[sys.executable,'/content/DRfold2/DRfold_infer.py',str(src['fasta']),str(scratch/'prediction')]
        return args,Path('/content/DRfold2'),os.environ.copy()
    if model=='Protenix':
        request=[{'name':target,'sequences':[{'rnaSequence':{'sequence':src['seq'],'count':1}}]}]
        jpath=scratch/(target+'.json'); json_save(jpath,request)
        args=[shutil.which('protenix'),'pred','-i',str(jpath),'-o',str(scratch/'prediction'),
              '-n','protenix_base_default_v1.0.0','-s','101','-c','10','-p','200',
              '-e','1','-d','fp32','--use_default_params','false',
              '--use_msa','false','--use_rna_msa','false','--use_template','false',
              '--trimul_kernel','torch','--triatt_kernel','torch',
              '--enable_cache','false','--enable_fusion','false','--enable_tf32','false']
        env=os.environ.copy()
        env['LAYERNORM_TYPE']='torch'
        return args,Path('/content'),env
    raise KeyError(model)

def find_selected_output(model,scratch):
    pred_dir=scratch/'prediction'
    if model=='trRosettaRNA2':
        return pred_dir/'model_1_relaxed200.pdb',None
    if model=='DRfold2':
        return pred_dir/'relax'/'model_1.pdb',None
    if model=='Protenix':
        cif=list(pred_dir.rglob('*_sample_0.cif'))
        conf=list(pred_dir.rglob('*_summary_confidence_sample_0.json'))
        assert len(cif)==len(conf)==1, f'Expected single Protenix CIF/confidence: {cif}, {conf}'
        return cif[0],conf[0]
    raise KeyError(model)

def existing_gpu_compute_pids():
    # Optional NVML process enumeration. Fail only when NVML actually reports a busy GPU.
    try:
        return sorted(set(int(p.pid) for p in pynvml.nvmlDeviceGetComputeRunningProcesses(GPU_HANDLE)))
    except Exception:
        return None  # not supported on some GPU/driver combinations

def run_target(model,target):
    old=verified_complete(model,target)
    if old is not None:
        print('SKIP verified GPU profiling:',model,target,old['peak_gpu_memory_mib_observed'],'MiB',flush=True)
        return True
    dest=run_root(model)/target
    attempt=next_attempt(dest)
    scratch=Path('/content/rna3d_vram_scratch')/model/target
    if scratch.exists():
        shutil.rmtree(scratch)  # ephemeral Colab scratch ONLY; never touches Drive
    scratch.mkdir(parents=True)
    log=scratch/'inference.log'
    trace=scratch/'gpu_memory_trace.csv.gz'
    print('START',model,target,INPUT[target]['length'],'nt |',utc_now(),flush=True)
    try:
        busy_pids=existing_gpu_compute_pids()
        if busy_pids:
            raise RuntimeError(f'GPU is already used by compute process(es) {busy_pids}. '
                               'Stop other GPU jobs before profiling to avoid contaminating VRAM figures.')
        args,cwd,env=prepare_command(model,target,scratch)
        ret,elapsed,telemetry=execute_monitored(args,cwd,log,trace,env)
        if ret!=0: raise RuntimeError(f'Inference exit code {ret}; inspect logged output')
        if telemetry['peak_gpu_memory_mib_above_baseline']<MIN_ACTIVE_GPU_DELTA_MIB:
            raise RuntimeError('GPU memory did not rise by at least 64 MiB during inference; '
                               'possible CPU fallback/monitoring failure. Not recording a false VRAM measurement.')
        output,confidence=find_selected_output(model,scratch)
        assert output.is_file() and output.stat().st_size>150, f'Missing selected structure: {output}'
        if model=='Protenix':
            validation=validate_cif(output,confidence,INPUT[target]['length'])
        else:
            validation=validate_pdb(output,INPUT[target]['length'])
        pred_name=RUNS[model]['prediction_filename']
        files={pred_name:output,'inference.log':log,'gpu_memory_trace.csv.gz':trace}
        if confidence is not None: files['summary_confidence.json']=confidence
        if model=='Protenix': files['input.json']=scratch/(target+'.json')
        else: files['input.'+('a3m' if model=='trRosettaRNA2' else 'fasta')]=INPUT[target]['a3m' if model=='trRosettaRNA2' else 'fasta']
        artifacts=[]
        for name,source in files.items():
            dest_file=attempt/name
            shutil.copy2(source,dest_file)
            assert sha256(dest_file)==sha256(source), f'Drive write verification failed: {name}'
            artifacts.append({'filename':name,'sha256':sha256(dest_file),'bytes':dest_file.stat().st_size})
        old_hash=prior_output_hash(model,target)
        pred_hash=sha256(attempt/pred_name)
        detail=dict(status='PASS',model=model,target_id=target,
                    model_commit=RUNS[model]['pin'],mode=RUNS[model]['mode'],
                    run_id=RUNS[model]['run_id'],attempt=attempt.name,
                    input_length=INPUT[target]['length'],fasta_sha256=INPUT[target]['fasta_sha256'],
                    a3m_sha256=INPUT[target]['a3m_sha256'] if model=='trRosettaRNA2' else None,
                    runtime_seconds=elapsed,preexisting_gpu_compute_pids=busy_pids,
                    prediction_sha256=pred_hash,
                    previous_prediction_sha256=old_hash,
                    byte_identical_to_previous=(pred_hash==old_hash) if old_hash is not None else None,
                    validation=validation,
                    inference_command=args,
                    observed_utc=utc_now(),
                    **telemetry)
        json_save(attempt/'peak_vram.json',detail,exclusive=True)
        artifacts.append({'filename':'peak_vram.json','sha256':sha256(attempt/'peak_vram.json'),
                          'bytes':(attempt/'peak_vram.json').stat().st_size})
        complete={k:v for k,v in detail.items() if k!='inference_command'}
        complete['artifacts']=artifacts
        # Canonical marker written LAST; completed attempts never overwritten.
        if (dest/'complete.json').exists():
            raise RuntimeError(f'{dest}/complete.json exists but was invalid: preserve evidence; inspect manually')
        json_save(dest/'complete.json',complete,exclusive=True)
        assert verified_complete(model,target) is not None, 'Post-save validation/read-back failed'
        print('PASS',model,target,'peak GPU used =',telemetry['peak_gpu_memory_mib_observed'],'MiB',
              '(above idle:',telemetry['peak_gpu_memory_mib_above_baseline'],'MiB)',flush=True)
        return True
    except Exception as e:
        diagnostic={'model':model,'target_id':target,'status':'FAIL','attempt':attempt.name,
                    'error':repr(e),'traceback':traceback.format_exc()[-12000:],'utc':utc_now()}
        json_save(attempt/'failed.json',diagnostic)
        for name,src in [('inference.log',log),('gpu_memory_trace.csv.gz',trace)]:
            if src.exists() and not (attempt/name).exists(): shutil.copy2(src,attempt/name)
        print('FAILED',model,target,str(e)[:450],'| Evidence:',attempt,flush=True)
        return False
    finally:
        if scratch.exists(): shutil.rmtree(scratch,ignore_errors=True)  # local-only cleanup

def update_summary():
    rows=[]
    for model in MODEL_ORDER:
        for target in TARGETS:
            complete=verified_complete(model,target)
            rows.append({
                'model':model,'target_id':target,'input_length':INPUT[target]['length'],
                'status':'PASS' if complete else 'NOT_VERIFIED',
                'peak_gpu_memory_mib_observed':complete['peak_gpu_memory_mib_observed'] if complete else '',
                'peak_gpu_memory_mib_above_baseline':complete['peak_gpu_memory_mib_above_baseline'] if complete else '',
                'baseline_gpu_memory_mib':complete['baseline_gpu_memory_mib'] if complete else '',
                'runtime_seconds':complete['runtime_seconds'] if complete else '',
                'sample_count':complete['sample_count'] if complete else '',
                'gpu_name':complete['gpu_name'] if complete else GPU_NAME,
                'sample_interval_ms':complete['sampling_interval_ms'] if complete else int(1000*SAMPLING_INTERVAL_SECONDS),
                'upstream_commit':RUNS[model]['pin'],
                'run_id':RUNS[model]['run_id'],
                'evidence_folder':str(run_root(model)/target),
                'measurement_method':'NVML sampled total device used' })
    path=EVAL/'peak_gpu_memory_30_targets.csv'
    with path.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]))
        writer.writeheader();writer.writerows(rows)
    with (EVAL/'peak_gpu_memory_30_targets.md').open('w') as f:
        f.write('# RNA3D Pilot10 — observed peak GPU memory (MiB)\n\n')
        f.write(f'Measurement: NVML sampled total GPU device used, every {SAMPLING_INTERVAL_SECONDS:.2f} s; Tesla T4. '
                'Values include baseline GPU use; the baseline-subtracted estimate is in the CSV. '
                'Only PASS values are experimentally measured.\n\n')
        f.write('| Target | trRosettaRNA2 | DRfold2 | Protenix |\n|---|---:|---:|---:|\n')
        idx={(x['model'],x['target_id']):x for x in rows}
        for t in TARGETS:
            values=[]
            for m in MODEL_ORDER:
                v=idx[(m,t)]['peak_gpu_memory_mib_observed']
                values.append(f'{float(v):.1f}' if v!='' else 'PENDING')
            f.write('| '+t+' | '+' | '.join(values)+' |\n')
        completed=sum(r['status']=='PASS' for r in rows)
        f.write(f'\n**Verified measurements: {completed}/30.**\n')
    return rows

print('Monitoring, status verification, and Drive persistence are ready.')


In [ ]:
# Cell 3/6 — trRosettaRNA2: pinned environment, weights, 10 automatic profiling runs

def setup_trrosetta(runlog):
    repo=Path('/content/trRosettaRNA2')
    pin=RUNS['trRosettaRNA2']['pin']
    if not (repo/'.git').is_dir():
        run_checked(['git','clone','https://github.com/YangLab-SDU/trRosettaRNA2.git',repo],log_file=runlog)
    actual=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
    if actual!=pin: run_checked(['git','-C',str(repo),'checkout','--detach',pin],log_file=runlog)
    assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==pin
    mm=Path('/content/bin/micromamba')
    if not mm.is_file():
        mm.parent.mkdir(parents=True,exist_ok=True)
        with urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest',timeout=120) as response:
            data=response.read()
        with tarfile.open(fileobj=io.BytesIO(data),mode='r:bz2') as tf:
            member=tf.extractfile('bin/micromamba')
            assert member is not None
            mm.write_bytes(member.read())
        mm.chmod(0o755)
    envroot='/content/micromamba_root'
    py=Path(envroot+'/envs/trRNA2/bin/python')
    if not py.exists():
        run_checked([mm,'-r',envroot,'create','-y','-n','trRNA2','-f','environment.yml'],cwd=repo,log_file=runlog)
    assert py.is_file()
    p=subprocess.run([str(py),'-c','import torch; print(torch.__version__,torch.cuda.is_available())'],
                     capture_output=True,text=True)
    assert p.returncode==0 and 'True' in p.stdout, f'trRNA2 CUDA environment not ready: {p.stderr[-800:]}'
    param=repo/'params'
    if not (param.exists() and (any(param.rglob('*.pt')) or any(param.rglob('*.pth')))):
        archive=repo/'params_trRNA2.tar.bz2'
        if not archive.is_file():
            mirrors=['https://huggingface.co/datasets/quailwwk/trRNA2/resolve/main/params_trRNA2.tar.bz2',
                     'http://yanglab.qd.sdu.edu.cn/trRosettaRNA/download/params_trRNA2.tar.bz2']
            errors=[]
            for url in mirrors:
                part=repo/'params_trRNA2.partial'
                try:
                    print('Downloading trRosettaRNA2 pretrained weights:',url,flush=True)
                    with urllib.request.urlopen(url,timeout=120) as source, part.open('wb') as dest:
                        shutil.copyfileobj(source,dest,length=2**20)
                    if not tarfile.is_tarfile(part): raise ValueError('Invalid weights archive')
                    part.rename(archive)
                    break
                except Exception as e:
                    errors.append(repr(e))
                    if part.exists(): part.unlink()
            else: raise RuntimeError('Weights mirrors failed: '+str(errors))
        run_checked(['tar','-xjf',archive],cwd=repo,log_file=runlog)
    assert (param.exists() and (any(param.rglob('*.pt')) or any(param.rglob('*.pth')))), 'trRNA2 weights absent after extraction'
    print('trRosettaRNA2 pinned setup PASS',flush=True)

def run_model(model,setup):
    root=lock_model_run(model)
    pending=[t for t in TARGETS if verified_complete(model,t) is None]
    if not pending:
        print(model,'already complete: 10/10; no inference repeated',flush=True)
        update_summary()
        return True
    try:
        setup(root/'setup_install.log')
    except Exception as e:
        folder=root/'setup_failures';folder.mkdir(parents=True,exist_ok=True)
        json_save(folder/('setup_failure_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.json'),
                  {'model':model,'utc':utc_now(),'error':repr(e),'traceback':traceback.format_exc()[-18000:]})
        print('SETUP FAILED',model,repr(e),'| See',root/'setup_install.log',flush=True)
        update_summary()
        return False
    for i,t in enumerate(TARGETS,1):
        print(f'[{model} {i}/10]',flush=True)
        try: run_target(model,t)
        except Exception as ex:
            print('UNEXPECTED TARGET ERROR:',model,t,repr(ex),flush=True)
        update_summary()   # Drive CSV updated after EACH target, even failed ones
    passed=sum(verified_complete(model,t) is not None for t in TARGETS)
    print('MODEL RESULT:',model,passed,'/10 valid VRAM measurements',flush=True)
    return passed==10

trRNA2_success=run_model('trRosettaRNA2',setup_trrosetta)


In [ ]:
# Cell 4/6 — DRfold2: pinned upstream, Arena/weights recovery and 10 automatic profiling runs

def setup_drfold2(runlog):
    repo=Path('/content/DRfold2')
    pin=RUNS['DRfold2']['pin']
    if not (repo/'.git').is_dir():
        run_checked(['git','clone','https://github.com/leeyang/DRfold2.git',repo],log_file=runlog)
    run_checked(['git','fetch','origin',pin],cwd=repo,log_file=runlog)
    run_checked(['git','checkout','--detach',pin],cwd=repo,log_file=runlog)
    assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==pin
    for package,mod in [('biopython','Bio'),('numpy','numpy'),('scipy','scipy')]:
        if importlib.util.find_spec(mod) is None:
            run_checked([sys.executable,'-m','pip','install',package],log_file=runlog)
    if not shutil.which('clang++'):
        run_checked(['apt-get','update','-qq'],log_file=runlog)
        run_checked(['apt-get','install','-y','clang'],log_file=runlog)
    assert shutil.which('clang++'), 'clang++ still missing; Arena compilation impossible'
    weights=repo/'model_hub'
    arena=repo/'Arena/Arena'
    chk=['cfg_95','cfg_96','cfg_97','cfg_99']
    have_weights=weights.is_dir() and all((weights/k).is_dir() and any((weights/k).iterdir()) for k in chk)
    if have_weights and (repo/'Arena/Makefile').exists() and not arena.exists():
        run_checked(['make','Arena'],cwd=repo/'Arena',log_file=runlog)
    elif not (have_weights and arena.exists()):
        run_checked(['bash','install.sh'],cwd=repo,log_file=runlog)
    assert arena.is_file() and os.access(arena,os.X_OK), 'DRfold2 Arena executable absent'
    assert all((weights/k).is_dir() and any((weights/k).iterdir()) for k in chk), 'DRfold2 weights missing'
    print('DRfold2 pinned setup PASS',flush=True)

drfold2_success=run_model('DRfold2',setup_drfold2)


In [ ]:
# Cell 5/6 — Protenix: corrected FP32 LayerNorm=torch, strict finite-geometry validation

def setup_protenix(runlog):
    pin=RUNS['Protenix']['pin']
    # Install AFTER other two models; this can update the active Colab Python packages.
    run_checked([sys.executable,'-m','pip','install','-q','gemmi',
                 'git+https://github.com/bytedance/Protenix.git@'+pin],log_file=runlog)
    import gemmi, importlib.metadata
    assert shutil.which('protenix'), 'Protenix CLI missing'
    print('Protenix',importlib.metadata.version('protenix'),'| gemmi',gemmi.__version__)
    print('Corrected configuration: RNA-only, fp32, LAYERNORM_TYPE=torch, seed=101, one sample, '
          '10 cycles, 200 diffusion steps; no MSA/templates/cache/fusion/TF32.',flush=True)

protenix_success=run_model('Protenix',setup_protenix)


In [ ]:
# Cell 6/6 — final Drive summary and explicit completeness gate
rows=update_summary()
passed=sum(r['status']=='PASS' for r in rows)
by_model={m:sum(r['status']=='PASS' and r['model']==m for r in rows) for m in MODEL_ORDER}
final={
    'experiment_id':EVAL.name,
    'status':'COMPLETE' if passed==30 else 'INCOMPLETE',
    'measured':passed,'expected':30,
    'models':by_model,
    'gpu_name':GPU_NAME,'gpu_total_memory_mib':GPU_TOTAL_MIB,
    'measurement_method':'NVML device-wide sampled total used; includes baseline GPU memory and CUDA subprocesses',
    'sample_interval_seconds':SAMPLING_INTERVAL_SECONDS,
    'minimum_required_gpu_delta_mib':MIN_ACTIVE_GPU_DELTA_MIB,
    'summary_csv':'peak_gpu_memory_30_targets.csv',
    'summary_markdown':'peak_gpu_memory_30_targets.md',
    'completed_utc':utc_now(),
    'source_original_runs_preserved':True,
    'note':'Experimentally recorded VRAM only; these values are not theoretical allocations or literature estimates. '
           'MSA/template conditions and model pins match the documented pilot; GPU memory traces are sampled, not exact instantaneous maxima.'
}
json_save(EVAL/'profiling_status.json',final)
print('=========================================',flush=True)
print('PROFILE RESULTS:',passed,'/30',by_model,flush=True)
print('STATUS:',final['status'],flush=True)
print('COMBINED CSV:',EVAL/'peak_gpu_memory_30_targets.csv',flush=True)
print('HUMAN TABLE:',EVAL/'peak_gpu_memory_30_targets.md',flush=True)
print('STATUS JSON:',EVAL/'profiling_status.json',flush=True)
print('=========================================',flush=True)
if passed!=30:
    print('Some targets are not verified. Review their setup_failures/ or attempt_*/failed.json '
          'and click Run all again on a T4 to resume; completed targets will be skipped.',flush=True)
else:
    print('SUCCESS: 30/30 verified GPU VRAM measurements saved to Google Drive.',flush=True)
